# Notebook 01 — Introdução e fluxo
Este notebook apresenta o pipeline do treinamento e os principais objetivos.


## Introdução ao treinamento

Este treinamento apresenta, de forma prática e progressiva, um fluxo completo de Ciência de Dados aplicado à caracterização e à predição de atributos físicos do solo. Ao longo dos notebooks, serão exploradas as principais etapas de um projeto de análise de dados, desde a importação e a avaliação da qualidade dos dados até a construção, a validação e a interpretação de modelos preditivos.

O trabalho utiliza dados espectrais e informações laboratoriais para demonstrar como técnicas de análise exploratória, pré-processamento, redução de dimensionalidade, clusterização e aprendizado de máquina podem apoiar a compreensão dos solos. Além de obter bons resultados, o treinamento enfatiza a organização do processo, a escolha adequada dos métodos e a interpretação dos resultados para que as conclusões sejam confiáveis e úteis.

Ao final, espera-se que você consiga acompanhar esse fluxo de maneira estruturada, identificar as decisões mais importantes em cada etapa e aplicar os conhecimentos adquiridos a novos conjuntos de dados.


## Fluxo de trabalho e principais objetivos

O treinamento seguirá um fluxo estruturado para transformar dados espectrais e análises laboratoriais em informações úteis sobre os atributos físicos do solo:

1. **Importação e organização dos dados**: reunir as leituras espectrais, as variáveis laboratoriais e os identificadores das amostras, garantindo que as fontes estejam corretamente relacionadas.
2. **Controle de qualidade (QC)**: verificar valores ausentes, duplicidades, inconsistências, unidades e possíveis outliers antes da análise.
3. **Análise exploratória (EDA)**: descrever as variáveis, observar suas distribuições, avaliar correlações e identificar padrões ou limitações nos dados.
4. **Pré-processamento espectral**: preparar os espectros por meio de limpeza, transformação, normalização ou seleção de variáveis, quando necessário.
5. **Redução de dimensionalidade**: utilizar PCA para resumir a informação espectral, visualizar a estrutura dos dados e investigar os principais eixos de variação.
6. **Clusterização**: agrupar amostras com características semelhantes e avaliar se os grupos representam padrões relevantes do solo.
7. **Modelagem preditiva**: desenvolver modelos para estimar atributos físicos a partir dos dados espectrais e selecionar estratégias adequadas ao problema.
8. **Validação e comparação**: medir o desempenho dos modelos com dados não utilizados no ajuste, comparando métricas e verificando sua capacidade de generalização.
9. **Interpretação dos resultados**: identificar as variáveis mais influentes, analisar os erros e usar ferramentas como o SHAP para explicar as previsões.
10. **Aplicação e comunicação**: organizar os resultados em conclusões claras, reconhecer as limitações do estudo e indicar como o fluxo pode ser aplicado a novos dados.

O principal objetivo é compreender não apenas como executar cada técnica, mas também como tomar decisões justificadas em cada etapa e construir um processo de análise confiável, reprodutível e útil para a caracterização do solo.


In [5]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Arquivo MIR disponível no workspace; ajuste este caminho para outro conjunto de dados.
DATA_FILE = Path(r"C:\Users\PC\Desktop\RESET\raw_mir_data_Saturin.csv")
TARGET_COLUMNS = ["Sand_gkg", "Silt_gkg", "Clay_gkg", "Density_kgdm3"]
EXCLUDE_COLUMNS = {
    "id", "ID", "ID_Unico", "Unnamed: 0", "X", "Y",
    "Upper_depth_cm", "Lower_depth_cm"
}
OUTPUT_DIR = Path("outputs_fluxo_solo")
RANDOM_STATE = 42
TEST_SIZE = 0.20


def is_spectral_column(column_name):
    """Identifica colunas de bandas por número de onda ou padrão espectral."""
    normalized_name = str(column_name).strip().lower().replace("-", "_")
    if re.search(r"(wavelength|wave|band|banda|spectr|nir|mir|swir|vis)", normalized_name):
        return True
    try:
        wavelength = float(normalized_name.replace(",", "."))
    except ValueError:
        return False
    return 300 <= wavelength <= 4000


if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {DATA_FILE.resolve()}. "
        "Atualize DATA_FILE para o caminho do CSV com espectros e atributos do solo."
    )

DATA = pd.read_csv(DATA_FILE, low_memory=False)
DATA = DATA.dropna(axis=1, how="all").drop_duplicates().reset_index(drop=True)

spectral_columns = [
    column for column in DATA.columns if is_spectral_column(column)
]
if len(spectral_columns) < 5:
    raise ValueError(
        "Foram encontradas menos de cinco colunas espectrais. "
        "Verifique os nomes das bandas ou ajuste is_spectral_column()."
    )

missing_targets = sorted(set(TARGET_COLUMNS) - set(DATA.columns))
if missing_targets:
    raise KeyError(f"Atributos-alvo ausentes no CSV: {missing_targets}")
target_columns = TARGET_COLUMNS

feature_data = DATA[spectral_columns].apply(pd.to_numeric, errors="coerce")
target_data = DATA[target_columns].apply(pd.to_numeric, errors="coerce")
valid_rows = target_data.notna().all(axis=1)
feature_data = feature_data.loc[valid_rows].reset_index(drop=True)
target_data = target_data.loc[valid_rows].reset_index(drop=True)
feature_data = feature_data.dropna(axis=1, how="all")
spectral_columns = feature_data.columns.tolist()

if len(feature_data) < 10:
    raise ValueError("São necessárias pelo menos 10 amostras completas para treinar e validar o modelo.")

X_train, X_test, y_train, y_test = train_test_split(
    feature_data, target_data, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

n_components = max(1, min(20, X_train.shape[0] - 1, X_train.shape[1]))
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=n_components, random_state=RANDOM_STATE)),
    ("regressor", RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])

model.fit(X_train, y_train)
predictions = pd.DataFrame(
    model.predict(X_test), columns=target_columns, index=y_test.index
)

metrics = pd.DataFrame([
    {
        "atributo": target,
        "R2": r2_score(y_test[target], predictions[target]),
        "RMSE": np.sqrt(mean_squared_error(y_test[target], predictions[target])),
        "MAE": mean_absolute_error(y_test[target], predictions[target]),
    }
    for target in target_columns
]).sort_values("R2", ascending=False)

# Agrupamento exploratório das amostras no espaço espectral reduzido.
transformed_spectra = model[:-1].transform(feature_data)
cluster_count = min(4, max(2, len(feature_data) // 20))
clusters = KMeans(n_clusters=cluster_count, random_state=RANDOM_STATE, n_init=10)
cluster_labels = clusters.fit_predict(transformed_spectra)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
metrics.to_csv(OUTPUT_DIR / "metricas_modelo.csv", index=False)
predictions.assign(**{f"observado_{target}": y_test[target] for target in target_columns}).to_csv(
    OUTPUT_DIR / "predicoes_validacao.csv", index=False
)
pd.DataFrame({"amostra": feature_data.index, "cluster": cluster_labels}).to_csv(
    OUTPUT_DIR / "clusters_espectrais.csv", index=False
)

print(f"Amostras utilizadas: {len(feature_data)}")
print(f"Bandas espectrais utilizadas: {len(spectral_columns)}")
print(f"Atributos previstos: {', '.join(target_columns)}")
print("\nDesempenho na validação:")
display(metrics)


Amostras utilizadas: 17402
Bandas espectrais utilizadas: 851
Atributos previstos: Sand_gkg, Silt_gkg, Clay_gkg, Density_kgdm3

Desempenho na validação:


,atributo,R2,RMSE,MAE
2,Clay_gkg,0.774571,74.675096,51.027428
0,Sand_gkg,0.758455,123.339005,93.289868
1,Silt_gkg,0.687392,103.936567,78.611961
3,Density_kgdm3,0.564405,0.179670,0.129438




Resultados:

- 17.402 amostras utilizadas;
- 851 bandas espectrais MIR;
- atributos previstos: `Sand_gkg`, `Silt_gkg`, `Clay_gkg` e `Density_kgdm3`;
- sem erros ou avisos na execução.

Desempenho:

| Atributo | R² |
|---|---:|
| Clay_gkg | 0,775 |
| Sand_gkg | 0,758 |
| Silt_gkg | 0,687 |
| Density_kgdm3 | 0,564 |

Arquivos gerados em:

`notebooks/outputs_fluxo_solo/`

- `metricas_modelo.csv`
- `predicoes_validacao.csv`
- `clusters_espectrais.csv`

Completed: *Restringir previsão aos atributos físicos* (2/3)



In [ ]:
# EDA espectral: ruído, clusters, regiões minerais e comportamento do carbono
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
DATA_FILE = Path(r"C:\Users\PC\Desktop\RESET\raw_mir_data_Saturin.csv")
if not DATA_FILE.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {DATA_FILE}")

# 1. Leitura robusta e identificação das bandas MIR.
for encoding in ("utf-8", "cp1252", "latin1"):
    try:
        spectral_data = pd.read_csv(DATA_FILE, encoding=encoding, low_memory=False)
        break
    except UnicodeDecodeError:
        continue
else:
    raise ValueError(f"Não foi possível decodificar {DATA_FILE.name}")

spectral_data = spectral_data.drop(
    columns=[column for column in spectral_data.columns if str(column).lower().startswith("unnamed")],
    errors="ignore",
).drop_duplicates().reset_index(drop=True)

wavelength_columns = []
for column in spectral_data.columns:
    try:
        wavelength = float(str(column).strip().replace(",", "."))
    except (TypeError, ValueError):
        continue
    if 300 <= wavelength <= 4000:
        wavelength_columns.append(column)

wavelength_columns = sorted(
    wavelength_columns,
    key=lambda column: float(str(column).strip().replace(",", ".")),
)
wavelengths = np.array(
    [float(str(column).strip().replace(",", ".")) for column in wavelength_columns],
    dtype=float,
)

if len(wavelength_columns) < 5:
    raise ValueError("Poucas bandas espectrais foram encontradas.")

spectra_raw = spectral_data[wavelength_columns].apply(pd.to_numeric, errors="coerce")
carbon = pd.to_numeric(spectral_data.get("C_gkg"), errors="coerce")
valid_rows = carbon.notna() & spectra_raw.notna().any(axis=1)
spectra_raw = spectra_raw.loc[valid_rows].reset_index(drop=True)
carbon = carbon.loc[valid_rows].reset_index(drop=True)
spectra_raw = spectra_raw.dropna(axis=1, how="all")
wavelengths = wavelengths[[column in spectra_raw.columns for column in wavelength_columns]]
wavelength_columns = spectra_raw.columns.tolist()

# 2. Controle de qualidade e interpolação por linha.
spectra = spectra_raw.to_numpy(dtype=float)
spectra[~np.isfinite(spectra)] = np.nan
for row_index in range(spectra.shape[0]):
    row = spectra[row_index]
    valid = np.isfinite(row)
    if valid.sum() == 0:
        row[:] = 0.0
    elif valid.sum() == 1:
        row[:] = row[valid][0]
    else:
        row[:] = np.interp(np.arange(row.size), np.flatnonzero(valid), row[valid])

# 3. Suavização Savitzky-Golay e perfil de ruído por amostra.
window_length = min(15, spectra.shape[1])
if window_length % 2 == 0:
    window_length -= 1
if window_length < 5:
    raise ValueError("São necessárias pelo menos cinco bandas para Savitzky-Golay.")

spectra_smooth = savgol_filter(
    spectra,
    window_length=window_length,
    polyorder=2,
    axis=1,
)
noise_signal = spectra - spectra_smooth
noise_profile = pd.DataFrame({
    "noise_mean": noise_signal.mean(axis=1),
    "noise_std": noise_signal.std(axis=1),
    "noise_rmse": np.sqrt(np.mean(noise_signal ** 2, axis=1)),
    "signal_std": spectra_smooth.std(axis=1),
})
noise_profile["snr_proxy"] = noise_profile["signal_std"] / np.maximum(noise_profile["noise_std"], 1e-12)

# 4. PCA e clusterização no espectro suavizado e padronizado.
scaled_spectra = StandardScaler().fit_transform(spectra_smooth)
n_components = min(10, scaled_spectra.shape[0] - 1, scaled_spectra.shape[1])
pca = PCA(n_components=n_components, random_state=RANDOM_STATE)
pca_scores = pca.fit_transform(scaled_spectra)
cluster_count = 4
cluster_model = KMeans(n_clusters=cluster_count, n_init=10, random_state=RANDOM_STATE)
cluster_labels = cluster_model.fit_predict(pca_scores[:, :min(10, n_components)])

# 5. Predição do carbono usando PLS no espectro suavizado.
train_index, test_index = train_test_split(
    np.arange(len(carbon)), test_size=0.20, random_state=RANDOM_STATE
)
carbon_scaler = StandardScaler()
train_spectra = carbon_scaler.fit_transform(spectra_smooth[train_index])
test_spectra = carbon_scaler.transform(spectra_smooth[test_index])
pls_components = min(15, train_spectra.shape[0] - 1, train_spectra.shape[1])
carbon_model = PLSRegression(n_components=pls_components, scale=False)
carbon_model.fit(train_spectra, carbon.iloc[train_index].to_numpy())
carbon_prediction = carbon_model.predict(test_spectra).ravel()
carbon_metrics = {
    "R2": r2_score(carbon.iloc[test_index], carbon_prediction),
    "RMSE": np.sqrt(mean_squared_error(carbon.iloc[test_index], carbon_prediction)),
    "MAE": mean_absolute_error(carbon.iloc[test_index], carbon_prediction),
}

# 6. Resumo do comportamento do carbono por cluster.
cluster_summary = pd.DataFrame({
    "cluster": cluster_labels,
    "C_gkg": carbon.to_numpy(),
    "noise_std": noise_profile["noise_std"].to_numpy(),
}).groupby("cluster").agg(
    samples=("C_gkg", "size"),
    carbon_mean_gkg=("C_gkg", "mean"),
    carbon_median_gkg=("C_gkg", "median"),
    carbon_std_gkg=("C_gkg", "std"),
    noise_std_mean=("noise_std", "mean"),
).reset_index()

# 7. Regiões espectrais candidatas para triagem mineralógica.
# São regiões de interesse para investigação, não uma identificação química definitiva.
mineral_regions = pd.DataFrame([
    {"region": "OH / água estrutural", "start_nm": 1350, "end_nm": 1450},
    {"region": "Água / OH combinado", "start_nm": 1850, "end_nm": 1950},
    {"region": "Argilominerais / OH", "start_nm": 2150, "end_nm": 2350},
    {"region": "Carbonatos / combinações", "start_nm": 2450, "end_nm": 2550},
    {"region": "Carbonatos / CO3", "start_nm": 3300, "end_nm": 3700},
])
region_rows = []
mean_spectrum = spectra_smooth.mean(axis=0)
for _, region in mineral_regions.iterrows():
    mask = (wavelengths >= region["start_nm"]) & (wavelengths <= region["end_nm"])
    if mask.any():
        region_rows.append({
            "region": region["region"],
            "start_nm": region["start_nm"],
            "end_nm": region["end_nm"],
            "mean_signal": mean_spectrum[mask].mean(),
            "min_signal": mean_spectrum[mask].min(),
            "max_signal": mean_spectrum[mask].max(),
        })
region_summary = pd.DataFrame(region_rows)

print("EDA ESPECTRAL MIR")
print("=" * 25)
print(f"Amostras: {spectra_smooth.shape[0]}")
print(f"Bandas: {spectra_smooth.shape[1]}")
print(f"Janela Savitzky-Golay: {window_length}")
print(f"Ruído médio (RMSE): {noise_profile['noise_rmse'].mean():.6f}")
print(f"SNR proxy mediano: {noise_profile['snr_proxy'].median():.3f}")
print(f"PCA PC1 + PC2: {pca.explained_variance_ratio_[:2].sum():.2%}")
print(f"Clusters gerados: {cluster_count}")
print(f"Carbono - R2: {carbon_metrics['R2']:.4f}")
print(f"Carbono - RMSE: {carbon_metrics['RMSE']:.4f} g/kg")
print(f"Carbono - MAE: {carbon_metrics['MAE']:.4f} g/kg")

print("\nComportamento do carbono por cluster:")
display(cluster_summary.round(4))
print("\nRegiões candidatas para interpretação mineralógica:")
display(region_summary.round(4))
print("\nAmostras com maior ruído relativo:")
display(noise_profile.sort_values("noise_rmse", ascending=False).head(10).round(6))

# 8. Visualizações: espectro, ruído, PCA/clusters e carbono observado versus previsto.
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

for cluster_id in range(cluster_count):
    cluster_mask = cluster_labels == cluster_id
    if cluster_mask.any():
        axes[0, 0].plot(wavelengths, spectra_smooth[cluster_mask].mean(axis=0), label=f"Cluster {cluster_id}")
axes[0, 0].set_title("Espectro suavizado médio por cluster")
axes[0, 0].set_xlabel("Comprimento de onda")
axes[0, 0].set_ylabel("Sinal")
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.25)

axes[0, 1].plot(wavelengths, noise_signal.mean(axis=0), color="firebrick")
axes[0, 1].axhline(0, color="black", linewidth=0.8)
axes[0, 1].set_title("Perfil médio do ruído: original - suavizado")
axes[0, 1].set_xlabel("Comprimento de onda")
axes[0, 1].set_ylabel("Resíduo espectral")
axes[0, 1].grid(alpha=0.25)

for cluster_id in range(cluster_count):
    cluster_mask = cluster_labels == cluster_id
    axes[1, 0].scatter(
        pca_scores[cluster_mask, 0],
        pca_scores[cluster_mask, 1],
        s=10,
        alpha=0.35,
        label=f"Cluster {cluster_id}",
    )
axes[1, 0].set_title("PCA e clusters espectrais")
axes[1, 0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
axes[1, 0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.25)

axes[1, 1].scatter(carbon.iloc[test_index], carbon_prediction, s=12, alpha=0.35)
plot_min = min(carbon.iloc[test_index].min(), carbon_prediction.min())
plot_max = max(carbon.iloc[test_index].max(), carbon_prediction.max())
axes[1, 1].plot([plot_min, plot_max], [plot_min, plot_max], "r--")
axes[1, 1].set_title("Carbono observado versus previsto")
axes[1, 1].set_xlabel("C observado (g/kg)")
axes[1, 1].set_ylabel("C previsto (g/kg)")
axes[1, 1].grid(alpha=0.25)

plt.tight_layout()
plt.show()

print(
    "Interpretação: clusters representam padrões de forma espectral; o perfil original-suavizado "
    "quantifica o ruído por banda e por amostra. As regiões minerais são candidatas para triagem "
    "e devem ser confirmadas com referências espectroscópicas e análises laboratoriais."
)
